# App-11b : Picross (Nonogrammes) — Jumeau C#

> **Twin C#** de [`App-11-Picross`](App-11-Picross.ipynb) (Python + OR-Tools CP-SAT + numpy + matplotlib).
> Suite du marathon **.NET ⇄ Python** (#4956), volet **Search / Applications / CSP**.
> BCL .NET seule, **0 NuGet**, **from-scratch** (Prong B #3801).

## Objectifs d'apprentissage

1. Modéliser un **nonogramme** (Picross) : indices par ligne/colonne = longueurs des blocs contigus.
2. Comprendre pourquoi le **backtracking naïf** (produit cartésien des motifs de ligne) explose combinatoirement.
3. ★ Implémenter la **propagation de contraintes ligne-par-ligne** : énumérer les motifs valides, les intersecter pour déduire les cellules **forcées**, itérer jusqu'au **point fixe**. C'est exactement la mécanique que CP-SAT automatise en interne — d'où le « speedup spectaculaire ».

## Complémentarité (#3801 Prong B)

| Twin | Outil | Valeur |
|------|-------|--------|
| Python | **OR-Tools CP-SAT** + numpy + matplotlib | solveur industriel, speedup mesuré |
| **Ce twin (.NET BCL seule)** | **from-scratch** (énumération de motifs, intersection, fixpoint) | **comprendre** la propagation de contraintes que CP-SAT fait sous le capot |

Le twin Python **appelle** CP-SAT et constate un speedup de plusieurs millions ; ce twin **déroule** l'algorithme de propagation qui produit ce speedup, rendant la mécanique explicite.

**Durée estimée : 40 min.** Prérequis : [`Search-9`](../../Part2-CSP/Search-9-ConstraintSatisfaction.ipynb) (CSP).


## 1. Représentation et énumération des motifs d'une ligne

Un puzzle Picross est défini par les **indices** de chaque ligne et colonne : la liste des longueurs des blocs de cellules remplies (contigües). Par exemple, l'indice `[2, 1]` sur une ligne de longueur 5 signifie : un bloc de 2, un bloc de 1, séparés par au moins une cellule vide.

Première brique : **énumérer tous les motifs valides** d'une ligne pour un indice donné (placement récursif des blocs).

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

static string FI(double x, string fmt = "F3") => x.ToString(fmt, CultureInfo.InvariantCulture);
static string FI(long x) => x.ToString("N0", CultureInfo.InvariantCulture);
static void Show(string s) => display(s);

// énumère TOUS les placements valides des blocs `runs` dans une ligne de longueur n
// retour : liste de bool[] (true = rempli). Cas spécial : indice vide ou [0] -> ligne toute vide.
static List<bool[]> EnumeratePatterns(int n, int[] runs) {
    var result = new List<bool[]>();
    var empty = new bool[n];
    if (runs.Length == 0 || (runs.Length == 1 && runs[0] == 0)) { result.Add(empty); return result; }
    void Rec(int runIdx, int startPos, bool[] line) {
        if (runIdx == runs.Length) { result.Add((bool[])line.Clone()); return; }
        int needed = 0;
        for (int i = runIdx; i < runs.Length; i++) needed += runs[i] + (i > runIdx ? 1 : 0);
        int lastStart = n - needed;
        for (int s = startPos; s <= lastStart; s++) {
            var line2 = (bool[])line.Clone();
            for (int k = 0; k < runs[runIdx]; k++) line2[s + k] = true;
            Rec(runIdx + 1, s + runs[runIdx] + 1, line2);
        }
    }
    Rec(0, 0, empty);
    return result;
}

static string PatternStr(bool[] p) => string.Join("", p.Select(b => b ? "#" : "."));

var pats = EnumeratePatterns(5, new[]{2, 1});
Show("Ligne n=5, indice [2,1] : " + pats.Count + " motifs valides");
foreach (var p in pats) Show("  " + PatternStr(p));
Show("Formule combinatoire : C(slack + nbBlocs, nbBlocs), slack = n - (somme blocs + gaps).");


Ligne n=5, indice [2,1] : 3 motifs valides

  ##.#.

  ##..#

  .##.#

Formule combinatoire : C(slack + nbBlocs, nbBlocs), slack = n - (somme blocs + gaps).

## 2. Déduction par intersection des motifs

Étant donné l'ensemble des motifs valides d'une ligne (éventuellement filtrés par les cellules **déjà connues**), on peut déduire les cellules **forcées** :

- une cellule est **forcément remplie** si elle l'est dans *tous* les motifs ;
- elle est **forcément vide** si elle est vide dans *tous* les motifs ;
- sinon elle reste **inconnue**.

C'est le cœur de la propagation.

In [2]:
// known[i] : -1 inconnu, 0 vide, 1 rempli. Filtre les motifs cohérents avec known,
// puis calcule l'intersection (forcé rempli / vide / inconnu). Retour null = contradiction.
static int[] LineDeduce(List<bool[]> patterns, int[] known) {
    int n = known.Length;
    var consistent = new List<bool[]>();
    foreach (var p in patterns) {
        bool ok = true;
        for (int i = 0; i < n; i++) {
            int v = p[i] ? 1 : 0;
            if (known[i] != -1 && known[i] != v) { ok = false; break; }
        }
        if (ok) consistent.Add(p);
    }
    if (consistent.Count == 0) return null;
    var deduced = new int[n];
    for (int i = 0; i < n; i++) {
        bool allFilled = true, allEmpty = true;
        foreach (var p in consistent) { if (!p[i]) allFilled = false; if (p[i]) allEmpty = false; }
        deduced[i] = allFilled ? 1 : (allEmpty ? 0 : -1);
    }
    return deduced;
}

// démo : ligne n=5 [2,1], intersection de TOUS les motifs (sans connaissance préalable)
var pats = EnumeratePatterns(5, new[]{2,1});
var known = new int[]{-1,-1,-1,-1,-1};
var ded = LineDeduce(pats, known);
Show("Ligne n=5 [2,1], intersection : " + string.Join(" ", ded.Select(d => d == 1 ? "#" : (d == 0 ? "." : "?"))));
Show("-> la cellule du milieu est FORCÉE remplie (présente dans les 3 motifs). Déduction de base.");


Ligne n=5 [2,1], intersection : ? # ? ? ?

-> la cellule du milieu est FORCÉE remplie (présente dans les 3 motifs). Déduction de base.

## 3. Solveur par propagation (point fixe)

On applique la déduction **alternativement sur toutes les lignes puis toutes les colonnes**, en propageant les cellules forcées, jusqu'à ce que plus rien ne change (**point fixe**). Pour les puzzles bien posés, cela suffit à résoudre toute la grille sans aucun retour arrière.

In [3]:
// grid[r][c] : -1 inconnu, 0 vide, 1 rempli. N lignes, M colonnes.
static (bool solved, int[][] grid, int iters, long patternsTried) SolvePropagate(int N, int M, int[][] rowClues, int[][] colClues) {
    var grid = new int[N][];
    for (int r = 0; r < N; r++) { grid[r] = new int[M]; for (int c = 0; c < M; c++) grid[r][c] = -1; }
    var rowPats = new List<bool[]>[N];
    for (int r = 0; r < N; r++) rowPats[r] = EnumeratePatterns(M, rowClues[r]);
    var colPats = new List<bool[]>[M];
    for (int c = 0; c < M; c++) colPats[c] = EnumeratePatterns(N, colClues[c]);
    long tried = 0;
    foreach (var lp in rowPats) tried += lp.Count;
    foreach (var lp in colPats) tried += lp.Count;
    int iters = 0; bool changed = true;
    while (changed) {
        changed = false; iters++;
        for (int r = 0; r < N; r++) {
            var kn = new int[M]; for (int c = 0; c < M; c++) kn[c] = grid[r][c];
            var d = LineDeduce(rowPats[r], kn);
            if (d == null) return (false, grid, iters, tried);
            for (int c = 0; c < M; c++) if (grid[r][c] != d[c]) { grid[r][c] = d[c]; changed = true; }
        }
        for (int c = 0; c < M; c++) {
            var kn = new int[N]; for (int r = 0; r < N; r++) kn[r] = grid[r][c];
            var d = LineDeduce(colPats[c], kn);
            if (d == null) return (false, grid, iters, tried);
            for (int r = 0; r < N; r++) if (grid[r][c] != d[r]) { grid[r][c] = d[r]; changed = true; }
        }
    }
    bool solved = true;
    for (int r = 0; r < N; r++) for (int c = 0; c < M; c++) if (grid[r][c] == -1) solved = false;
    return (solved, grid, iters, tried);
}

static string GridStr(int[][] g) {
    var sb = new System.Text.StringBuilder();
    for (int r = 0; r < g.Length; r++) {
        for (int c = 0; c < g[0].Length; c++) sb.Append(g[r][c] == 1 ? "##" : (g[r][c] == 0 ? ".." : "??"));
        sb.AppendLine();
    }
    return sb.ToString();
}

// dériver les clues depuis une solution construite (garantie cohérentes)
static (int[][] rc, int[][] cc) DeriveClues(bool[][] sol) {
    int N = sol.Length, M = sol[0].Length;
    var rc = new int[N][];
    for (int r = 0; r < N; r++) {
        var runs = new List<int>(); int cur = 0;
        for (int c = 0; c < M; c++) { if (sol[r][c]) cur++; else { if (cur > 0) runs.Add(cur); cur = 0; } }
        if (cur > 0) runs.Add(cur);
        rc[r] = runs.Count == 0 ? new[]{0} : runs.ToArray();
    }
    var cc = new int[M][];
    for (int c = 0; c < M; c++) {
        var runs = new List<int>(); int cur = 0;
        for (int r = 0; r < N; r++) { if (sol[r][c]) cur++; else { if (cur > 0) runs.Add(cur); cur = 0; } }
        if (cur > 0) runs.Add(cur);
        cc[c] = runs.Count == 0 ? new[]{0} : runs.ToArray();
    }
    return (rc, cc);
}

// diamant NxN (solution construite) : cellule remplie si |r-c| + |col-c| <= c, c = N/2
static bool[][] Diamond(int N) {
    int mid = N / 2;
    var sol = new bool[N][];
    for (int r = 0; r < N; r++) { sol[r] = new bool[N]; for (int col = 0; col < N; col++) sol[r][col] = Math.Abs(r - mid) + Math.Abs(col - mid) <= mid; }
    return sol;
}

var sol5 = Diamond(5);
var (rc5, cc5) = DeriveClues(sol5);
var (s5, g5, it5, tr5) = SolvePropagate(5, 5, rc5, cc5);
Show("Puzzle diamant 5x5 — propagation : resolu=" + s5 + " en " + it5 + " iterations (" + FI(tr5) + " motifs eumeres).");
Show(GridStr(g5));


Puzzle diamant 5x5 — propagation : resolu=True en 3 iterations (34 motifs eumeres).

....##....
..######..
##########
..######..
....##....


## 4. Approche naïve : produit cartésien (baseline exponentielle)

À l'opposé, le backtracking naïf : on prend **un motif par ligne** (produit cartésien) et on vérifie si les colonnes sont satisfaites. Sur une grille 5×5 avec ~3-5 motifs/ligne, cela fait quelques centaines de combinaisons — jouable. Mais la taille explose : 15×15 avec des dizaines de motifs/ligne → un produit astronomique (inatteignable).

C'est la baseline qui met en évidence la valeur de la propagation.

In [4]:
// backtracking naïf : choisit un motif par ligne (DFS), valide toutes les colonnes à la fin
static (bool solved, int[][] grid, long combos) SolveNaive(int N, int M, int[][] rowClues, int[][] colClues, long comboLimit) {
    var rowPats = new List<bool[]>[N];
    for (int r = 0; r < N; r++) rowPats[r] = EnumeratePatterns(M, rowClues[r]);
    var chosen = new bool[N][];
    long combos = 0;
    bool ColOk() {
        for (int c = 0; c < M; c++) {
            var runs = new List<int>(); int cur = 0;
            for (int r = 0; r < N; r++) { if (chosen[r][c]) cur++; else { if (cur > 0) runs.Add(cur); cur = 0; } }
            if (cur > 0) runs.Add(cur);
            var clue = colClues[c]; bool isEmpty = clue.Length == 1 && clue[0] == 0;
            if (isEmpty) { if (runs.Count != 0) return false; }
            else { if (runs.Count != clue.Length) return false; for (int i = 0; i < clue.Length; i++) if (runs[i] != clue[i]) return false; }
        }
        return true;
    }
    bool Rec(int r) {
        if (r == N) return ColOk();
        foreach (var p in rowPats[r]) {
            chosen[r] = p; combos++;
            if (combos > comboLimit) return false;
            if (Rec(r + 1)) return true;
        }
        return false;
    }
    if (Rec(0)) {
        var g = new int[N][]; for (int r = 0; r < N; r++) { g[r] = new int[M]; for (int c = 0; c < M; c++) g[r][c] = chosen[r][c] ? 1 : 0; }
        return (true, g, combos);
    }
    return (false, null, combos);
}

var (rc, cc) = DeriveClues(Diamond(5));
var (ok, g, combos) = SolveNaive(5, 5, rc, cc, 100_000_000);
Show("Diamant 5x5 — naïf : resolu=" + ok + " apres " + FI(combos) + " combinaisons testees.");


Diamant 5x5 — naïf : resolu=True apres 155 combinaisons testees.

## 5. ★ Le speedup spectaculaire : naïf vs propagation

On compare, pour des grilles de taille croissante :
- le **produit cartésien théorique** (∏ motifs par ligne) que le naïf devrait explorer — il devient astronomique ;
- la **somme des motifs énumérés** par la propagation (∑ motifs par ligne + colonne) — reste gérable ;
- pour le 5×5 seulement, le **compte réel** de combinaisons testées par le naïf.

Le naïf explose exponentiellement ; la propagation reste polynomiale. C'est le même enseignement que le « facteur plusieurs millions » mesuré par CP-SAT dans le twin Python — ici déroulé from-scratch.

In [5]:
Show("Taille".PadLeft(7) + " | " + "prod cartesien".PadLeft(16) + " | " + "propa (somme)".PadLeft(15) + " | " + "propa iters".PadLeft(12) + " | " + "naif reel 5x5".PadLeft(15));
Show(new string('-', 72));
foreach (var N in new[]{5, 10, 15}) {
    var (rc, cc) = DeriveClues(Diamond(N));
    var (solved, grid, iters, tried) = SolvePropagate(N, N, rc, cc);
    double prod = 1; double sumP = 0;
    for (int r = 0; r < N; r++) { int cnt = EnumeratePatterns(N, rc[r]).Count; prod *= cnt; sumP += cnt; }
    for (int c = 0; c < N; c++) sumP += EnumeratePatterns(N, cc[c]).Count;
    string prodCell = prod > 1e15 ? prod.ToString("0.0E0", CultureInfo.InvariantCulture) : FI((long)prod);
    string naiveReel = "-";
    if (N == 5) { var (ok, g, comb) = SolveNaive(N, N, rc, cc, 100_000_000); naiveReel = FI(comb); }
    Show((N + "x" + N).PadLeft(7) + " | " + prodCell.PadLeft(16) + " | " + FI((long)sumP).PadLeft(15) + " | " + iters.ToString().PadLeft(12) + " | " + naiveReel.PadLeft(15));
}
Show("Interpretation : le naif devrait explorer un produit cartesien exponentiel ;");
Show("la propagation n'enumere qu'une fois les motifs par ligne/colonne et les intersecte.");
Show("Le speedup croit exponentiellement avec la taille.");


 Taille |   prod cartesien |   propa (somme) |  propa iters |   naif reel 5x5

------------------------------------------------------------------------

    5x5 |              225 |              34 |            3 |             155

  10x10 |        1,474,560 |             102 |            4 |               -

  15x15 | 4,108,830,350,625 |             254 |            5 |               -

Interpretation : le naif devrait explorer un produit cartesien exponentiel ;

la propagation n'enumere qu'une fois les motifs par ligne/colonne et les intersecte.

Le speedup croit exponentiellement avec la taille.

## 6. Résolution d'un puzzle réaliste (15×15)

Sur un 15×15, le naïf est inatteignable (produit cartésien astronomique). La propagation résout en quelques itérations.

In [6]:
var sol15 = Diamond(15);
var (rc15, cc15) = DeriveClues(sol15);
var sw = System.Diagnostics.Stopwatch.StartNew();
var (solved, grid, iters, tried) = SolvePropagate(15, 15, rc15, cc15);
sw.Stop();
Show("Diamant 15x15 — propagation : resolu=" + solved + " en " + iters + " iterations / " + FI(sw.Elapsed.TotalMilliseconds, "F2") + " ms (" + FI(tried) + " motifs).");
Show(GridStr(grid));
Show("Verification : la grille reconstruite correspond a la solution cible -> " + (solved ? "OK" : "RESIDUEL (propagation incomplete, voir Exercice 3)."));


Diamant 15x15 — propagation : resolu=True en 5 iterations / 0.26 ms (254 motifs).

..............##..............
............######............
..........##########..........
........##############........
......##################......
....######################....
..##########################..
##############################
..##########################..
....######################....
......##################......
........##############........
..........##########..........
............######............
..............##..............


Verification : la grille reconstruite correspond a la solution cible -> OK

### 6.1 Unicité de la solution

Un bon puzzle a une solution **unique**. Si la propagation résout totalement la grille (plus aucune cellule inconnue au point fixe), l'unicité est garantie : il n'y a qu'une seule affectation cohérente avec tous les indices.

In [7]:
// compte les solutions via propagation + backtracking résiduel (plafonné)
static int CountSolutions(int N, int M, int[][] rowClues, int[][] colClues, int cap) {
    var (propSolved, grid, it, tried) = SolvePropagate(N, M, rowClues, colClues);
    if (propSolved) return 1;
    var rowPats = new List<bool[]>[N];
    for (int r = 0; r < N; r++) rowPats[r] = EnumeratePatterns(M, rowClues[r]);
    var chosen = new bool[N][];
    int count = 0;
    bool ColOk() {
        for (int c = 0; c < M; c++) {
            var runs = new List<int>(); int cur = 0;
            for (int r = 0; r < N; r++) { if (chosen[r][c]) cur++; else { if (cur > 0) runs.Add(cur); cur = 0; } }
            if (cur > 0) runs.Add(cur);
            var clue = colClues[c]; bool isEmpty = clue.Length == 1 && clue[0] == 0;
            if (isEmpty) { if (runs.Count != 0) return false; }
            else { if (runs.Count != clue.Length) return false; for (int i = 0; i < clue.Length; i++) if (runs[i] != clue[i]) return false; }
        }
        return true;
    }
    void Rec(int r) {
        if (count >= cap) return;
        if (r == N) { if (ColOk()) count++; return; }
        foreach (var p in rowPats[r]) { chosen[r] = p; Rec(r + 1); if (count >= cap) return; }
    }
    Rec(0);
    return count;
}

var (rc, cc) = DeriveClues(Diamond(5));
int sols = CountSolutions(5, 5, rc, cc, 5);
Show("Diamant 5x5 : " + sols + " solution(s) [cap 5]. " + (sols == 1 ? "Solution unique." : "Solution NON unique."));


Diamant 5x5 : 1 solution(s) [cap 5]. Solution unique.

## Synthèse

| Approche | Stratégie | Complexité | Quand l'utiliser |
|----------|-----------|------------|------------------|
| Naïf (produit cartésien) | un motif/ligne, valide colonnes | exponentiel (∏ motifs) | petites grilles, baseline |
| **Propagation (intersection + fixpoint)** | déduire cellules forcées, itérer | polynomial par itération (∑ motifs) | **toujours** (résout la plupart des puzzles) |

**Leçon** : la propagation de contraintes est ce qui transforme un problème exponentiel en problème traitable. CP-SAT (twin Python) industrialise cette mécanique (propagation + learning + branchement) ; ce twin la **déroule à la main** pour qu'elle soit visible. Le « speedup spectaculaire » n'est pas de la magie : c'est l'élimination du produit cartésien au profit de la déduction locale itérée.

In [8]:
Show("Synthese affichee ci-dessus (tableau markdown).");

Synthese affichee ci-dessus (tableau markdown).

## 8. Exercices

### Exercice 1 : énumérer les motifs d'un indice complexe

Pour une ligne de longueur 10 avec l'indice `[3, 2, 1]`, combien de motifs valides ? Comparer au résultat de `EnumeratePatterns` et vérifier la formule `C(slack + blocs, blocs)`.

> **Indice** : slack = 10 − (3+2+1) − 2 gaps = 2 ; blocs = 3 ; C(2+3, 3) = 10.

In [9]:
// Exercice 1 : compter les motifs de [3,2,1] sur n=10 et vérifier C(slack+blocs, blocs)
// Etape 1 : EnumeratePatterns(10, [3,2,1]).Count.
// Etape 2 : calculer C(2+3,3)=10 à la main, comparer.
static long Binomial(int n, int k) { long r = 1; for (int i = 0; i < k; i++) { r = r * (n - i) / (i + 1); } return r; }
Show("Exercice 1 — attendu : " + Binomial(5, 3) + " motifs (slack=2, blocs=3).");


Exercice 1 — attendu : 10 motifs (slack=2, blocs=3).

### Exercice 2 : encoder une image, déduire les indices, résoudre

Construire la solution cible d'une lettre « H » sur une grille 10×10 (bool[10][10]), en déduire les indices via `DeriveClues`, puis résoudre par propagation. Vérifier que la grille reconstruite correspond à la cible.

> **Indice** : `DeriveClues(h)` renvoie `(rc, cc)` ; lancer `SolvePropagate(10, 10, rc, cc)` ; comparer `grid` à `h`.

In [10]:
// Exercice 2 : encoder "H" en grille 10x10, déduire indices, résoudre par propagation
// Etape 1 : bool[][] h = ... (dessiner le H).
// Etape 2 : var (rc, cc) = DeriveClues(h).
// Etape 3 : var (ok, g, it, tr) = SolvePropagate(10, 10, rc, cc) -> comparer g à h.
Show("Exercice 2 à compléter — encoder une image, déduire les indices, résoudre.");


Exercice 2 à compléter — encoder une image, déduire les indices, résoudre.

### Exercice 3 : propagation incomplète → backtracking

Certains puzzles ne sont pas entièrement résolus par propagation seule (cellules inconnues subsistant au fixpoint). Compléter le solveur : quand la propagation stalle, choisir une cellule inconnue, **brancher** (essayer rempli puis vide), re-propager. C'est le schéma « propagation + recherche » qu'utilise CP-SAT.

> **Indice** : après `SolvePropagate`, si `solved == false`, trouver la première cellule `-1`, la forcer à 1 puis 0, rappeler `SolvePropagate` récursivement (en propageant la contrainte ajoutée).

In [11]:
// Exercice 3 : SolvePropagate + backtracking sur cellules résiduelles
// Etape 1 : détecter le stall (solved=false sans contradiction).
// Etape 2 : brancher sur la 1re cellule inconnue (1 puis 0), re-propager.
// Etape 3 : tester sur un puzzle ambigu (plusieurs solutions).
Show("Exercice 3 à compléter — propagation + branchement sur résiduel.");


Exercice 3 à compléter — propagation + branchement sur résiduel.

## Conclusion

Ce twin C# a **déroulé from-scratch** la mécanique de résolution des nonogrammes :

- **Énumération des motifs** valides d'une ligne (placement récursif des blocs) ;
- **Intersection** des motifs pour déduire les cellules forcées ;
- **Propagation** ligne/colonne jusqu'au point fixe ;
- **Comparaison** avec le backtracking naïf (baseline exponentielle) → le speedup.

★ **Leçon** : la propagation de contraintes (déduction locale itérée) transforme un problème exponentiel en problème polynomial — c'est exactement ce que CP-SAT industrialise, et dont le « speedup spectaculaire » du twin Python n'est que la manifestation mesurée.

**Lien avec les Foundations** : CSP ⇄ [`Search-9`](../../Part2-CSP/Search-9-ConstraintSatisfaction.ipynb), propagation ⇄ [`Search-9` §consistance arc](../../Part2-CSP/Search-9-ConstraintSatisfaction.ipynb), CP-SAT ⇄ [`App-11-Picross`](App-11-Picross.ipynb) (twin Python, solveur industriel).

---
*Marathon .NET ⇄ Python #4956 — Search/Applications, tranche Picross / nonogrammes. See #4956, #3801.*
